In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [2]:
FEATURES = [
    "backward_count",
    "replay_count",
    "study_pause_count",
    "other_work_pause_count",
    "forward_count",
    "speed_increase_count"
]

CLASSES = [
    "HIGH",
    "LOW",
    "MEDIUM"
]

print(FEATURES)

['backward_count', 'replay_count', 'study_pause_count', 'other_work_pause_count', 'forward_count', 'speed_increase_count']


In [3]:
np.random.seed(42)

n_samples = 3000

training_data = pd.DataFrame({
    "backward_count": np.random.poisson(3.2, n_samples),
    "replay_count": np.random.poisson(3.3, n_samples),
    "study_pause_count": np.random.poisson(2.7, n_samples),
    "other_work_pause_count": np.random.poisson(2.0, n_samples),
    "forward_count": np.random.poisson(2.5, n_samples),
    "speed_increase_count": np.random.poisson(1.5, n_samples)
})

training_data[FEATURES] = training_data[FEATURES].clip(lower=0)

In [4]:
difficulty_signal = (
    training_data["backward_count"] * 2.0 +
    training_data["replay_count"] * 2.0 +
    training_data["study_pause_count"] * 2.5 -
    training_data["other_work_pause_count"] * 1.0 -
    training_data["forward_count"] * 1.0 -
    training_data["speed_increase_count"] * 0.8
)

low_threshold = difficulty_signal.quantile(0.33)
high_threshold = difficulty_signal.quantile(0.66)

training_data["quiz_priority"] = np.where(
    difficulty_signal >= high_threshold,
    "HIGH",
    np.where(
        difficulty_signal <= low_threshold,
        "LOW",
        "MEDIUM"
    )
)

print(training_data["quiz_priority"].value_counts())

quiz_priority
HIGH      1040
LOW        993
MEDIUM     967
Name: count, dtype: int64


In [5]:
X = training_data[FEATURES]
y = training_data["quiz_priority"]

print(X.shape)
print(y.shape)

(3000, 6)
(3000,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (2400, 6)
Testing: (600, 6)


In [7]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression trained successfully")

Logistic Regression trained successfully


In [8]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

gradient_boosting = GradientBoostingClassifier(
    random_state=42
)

random_forest.fit(X_train, y_train)
gradient_boosting.fit(X_train, y_train)

print("Random Forest trained successfully")
print("Gradient Boosting trained successfully")

Random Forest trained successfully
Gradient Boosting trained successfully


In [9]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

gradient_boosting = GradientBoostingClassifier(
    random_state=42
)

random_forest.fit(X_train, y_train)
gradient_boosting.fit(X_train, y_train)

print("Random Forest trained successfully")
print("Gradient Boosting trained successfully")

Random Forest trained successfully
Gradient Boosting trained successfully


In [10]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest,
    "Gradient Boosting": gradient_boosting
}

results = []

for name, model in models.items():

    prediction = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(
            y_test,
            prediction,
            average="weighted"
        ),
        "Recall": recall_score(
            y_test,
            prediction,
            average="weighted"
        ),
        "F1 Score": f1_score(
            y_test,
            prediction,
            average="weighted"
        )
    })

results_df = pd.DataFrame(results)

display(results_df)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.996667,0.996701,0.996667,0.996667
1,Random Forest,0.858333,0.859439,0.858333,0.858521
2,Gradient Boosting,0.870000,0.872724,0.870000,0.870990


In [11]:
final_model = logistic_model

print("Final model: Logistic Regression")
print("Classes:", final_model.classes_)

Final model: Logistic Regression
Classes: ['HIGH' 'LOW' 'MEDIUM']


In [12]:
prediction = final_model.predict(X_test)

print(
    classification_report(
        y_test,
        prediction
    )
)

              precision    recall  f1-score   support

        HIGH       1.00      0.99      1.00       208
         LOW       1.00      1.00      1.00       199
      MEDIUM       0.99      1.00      0.99       193

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600



In [13]:
cm = confusion_matrix(
    y_test,
    prediction,
    labels=final_model.classes_
)

print(cm)

[[206   0   2]
 [  0 199   0]
 [  0   0 193]]


In [14]:
model_package = {
    "model": final_model,
    "feature_names": FEATURES,
    "classes": list(final_model.classes_),
    "version": "1.0"
}

joblib.dump(
    model_package,
    "adaptive_quiz_model.pkl"
)

print("adaptive_quiz_model.pkl saved successfully")

adaptive_quiz_model.pkl saved successfully


In [15]:
package = joblib.load(
    "adaptive_quiz_model.pkl"
)

model = package["model"]
FEATURES = package["feature_names"]

print("Model loaded")
print("Features:", FEATURES)

Model loaded
Features: ['backward_count', 'replay_count', 'study_pause_count', 'other_work_pause_count', 'forward_count', 'speed_increase_count']


In [16]:
def load_student_datasets(
    difficulty_path,
    telemetry_path,
    mapping_path
):

    difficulty = pd.read_csv(
        difficulty_path
    )

    telemetry = pd.read_csv(
        telemetry_path
    )

    mapping = pd.read_csv(
        mapping_path
    )

    return (
        difficulty,
        telemetry,
        mapping
    )

In [17]:
def get_student_data(
    difficulty,
    telemetry,
    mapping,
    student_id,
    course_name
):

    difficulty_student = difficulty[
        (difficulty["StudentID"] == student_id) &
        (difficulty["CourseName"] == course_name)
    ].copy()

    telemetry_student = telemetry[
        (telemetry["StudentID"] == student_id) &
        (telemetry["CourseName"] == course_name)
    ].copy()

    mapping_student = mapping[
        (mapping["StudentID"] == student_id) &
        (mapping["CourseName"] == course_name)
    ].copy()

    return (
        difficulty_student,
        telemetry_student,
        mapping_student
    )

In [18]:
def map_events_to_concepts(
    telemetry,
    mapping
):

    keys = [
        "StudentID",
        "CourseSubject",
        "CourseName",
        "EventType",
        "EventTimestamp"
    ]

    mapped = telemetry.merge(
        mapping[
            keys +
            [
                "Concept",
                "Transcript",
                "Prerequisite"
            ]
        ],
        on=keys,
        how="left"
    )

    return mapped

In [19]:
def normalize_pause_reason(reason):

    if pd.isna(reason):
        return None

    reason = str(reason).lower().strip()

    if reason == "study":
        return "study"

    if reason == "other_work":
        return "other_work"

    return None

In [20]:
def generate_concept_features(
    telemetry,
    mapping
):

    events = map_events_to_concepts(
        telemetry,
        mapping
    )

    events["NormalizedPauseReason"] = (
        events["PauseReason"]
        .apply(normalize_pause_reason)
    )

    events["backward_count"] = (
        events["EventType"]
        .eq("SEEK_BACKWARD")
        .astype(int)
    )

    events["replay_count"] = (
        events["EventType"]
        .eq("REPLAY")
        .astype(int)
    )

    events["study_pause_count"] = (
        events["NormalizedPauseReason"]
        .eq("study")
        .astype(int)
    )

    events["other_work_pause_count"] = (
        events["NormalizedPauseReason"]
        .eq("other_work")
        .astype(int)
    )

    events["forward_count"] = (
        events["EventType"]
        .eq("SEEK_FORWARD")
        .astype(int)
    )

    events["speed_increase_count"] = (
        (
            events["EventType"]
            .eq("SPEED_CHANGE")
        )
        &
        (
            pd.to_numeric(
                events["PlaybackSpeed"],
                errors="coerce"
            ) > 1
        )
    ).astype(int)

    features = (
        events
        .groupby(
            [
                "StudentID",
                "CourseName",
                "Concept"
            ]
        )[FEATURES]
        .sum()
        .reset_index()
    )

    return features

In [21]:
def predict_quiz_priority(
    concept_features
):

    X_live = concept_features[
        FEATURES
    ]

    predictions = model.predict(
        X_live
    )

    probabilities = model.predict_proba(
        X_live
    )

    concept_features = concept_features.copy()

    concept_features[
        "quiz_priority"
    ] = predictions

    concept_features[
        "confidence"
    ] = probabilities.max(
        axis=1
    )

    return concept_features

In [22]:
QUESTION_COUNT = {
    "HIGH": 5,
    "MEDIUM": 3,
    "LOW": 1
}

def assign_question_count(
    concept_results
):

    concept_results = concept_results.copy()

    concept_results[
        "question_count"
    ] = concept_results[
        "quiz_priority"
    ].map(
        QUESTION_COUNT
    )

    return concept_results

In [23]:
def generate_adaptive_quiz_priority(
    difficulty,
    telemetry,
    mapping,
    student_id,
    course_name
):

    difficulty_student, telemetry_student, mapping_student = get_student_data(
        difficulty,
        telemetry,
        mapping,
        student_id,
        course_name
    )

    concept_features = generate_concept_features(
        telemetry_student,
        mapping_student
    )

    results = predict_quiz_priority(
        concept_features
    )

    results = assign_question_count(
        results
    )

    return results

In [26]:
from google.colab import files

uploaded = files.upload()

Saving difficulty.csv to difficulty.csv
Saving interaction_telemetry (6).csv to interaction_telemetry (6).csv
Saving concept_event_mapping (3).csv to concept_event_mapping (3).csv


In [28]:
import pandas as pd

difficulty = pd.read_csv("difficulty.csv")
telemetry = pd.read_csv("interaction_telemetry (6).csv")
mapping = pd.read_csv("concept_event_mapping (3).csv")

print("Difficulty:", difficulty.shape)
print("Telemetry:", telemetry.shape)
print("Mapping:", mapping.shape)

Difficulty: (7, 13)
Telemetry: (123, 9)
Mapping: (123, 9)


In [29]:
print("DIFFICULTY COLUMNS:")
print(difficulty.columns.tolist())

print("\nTELEMETRY COLUMNS:")
print(telemetry.columns.tolist())

print("\nMAPPING COLUMNS:")
print(mapping.columns.tolist())

DIFFICULTY COLUMNS:
['StudentID', 'StudentName', 'CourseSubject', 'CourseName', 'Concept', 'ReplayCount', 'PauseCount', 'SeekBackwardCount', 'SeekForwardCount', 'UnderstandPauseCount', 'DifficultyScore', 'DifficultyLevel', 'NeedsRevision']

TELEMETRY COLUMNS:
['StudentID', 'StudentName', 'CourseSubject', 'CourseName', 'EventType', 'EventTimestamp', 'EventDuration', 'PlaybackSpeed', 'PauseReason']

MAPPING COLUMNS:
['StudentID', 'StudentName', 'CourseSubject', 'CourseName', 'EventType', 'EventTimestamp', 'Transcript', 'Concept', 'Prerequisite']


In [30]:
print("Students in difficulty:")
print(difficulty["StudentID"].unique())

print("\nCourses in difficulty:")
print(difficulty["CourseName"].unique())

print("\nStudents in telemetry:")
print(telemetry["StudentID"].unique())

print("\nCourses in telemetry:")
print(telemetry["CourseName"].unique())

print("\nStudents in mapping:")
print(mapping["StudentID"].unique())

print("\nCourses in mapping:")
print(mapping["CourseName"].unique())

Students in difficulty:
['S24A003']

Courses in difficulty:
['Intoduction to ML' 'Well posed problem' 'perspectives']

Students in telemetry:
['S24A003']

Courses in telemetry:
['perspectives' 'Well posed problem' 'Intoduction to ML']

Students in mapping:
['S24A003']

Courses in mapping:
['Intoduction to ML' 'Well posed problem' 'perspectives']


In [31]:
result = generate_adaptive_quiz_priority(
    difficulty,
    telemetry,
    mapping,
    student_id="S24A003",
    course_name="Intoduction to ML"
)

display(result)

,StudentID,CourseName,Concept,backward_count,replay_count,study_pause_count,other_work_pause_count,forward_count,speed_increase_count,quiz_priority,confidence,question_count
0,S24A003,Intoduction to ML,Intoduction To Ml Fundamentals,13,0,0,0,4,1,HIGH,0.999982,5
1,S24A003,Intoduction to ML,Introduction & Concepts,1,0,0,0,1,0,LOW,1.000000,1
2,S24A003,Intoduction to ML,Search Fundamentals,2,0,0,0,1,0,LOW,1.000000,1


In [32]:
result = generate_adaptive_quiz_priority(
    difficulty,
    telemetry,
    mapping,
    student_id="S24A003",
    course_name="Well posed problem"
)

display(result)

,StudentID,CourseName,Concept,backward_count,replay_count,study_pause_count,other_work_pause_count,forward_count,speed_increase_count,quiz_priority,confidence,question_count
0,S24A003,Well posed problem,Well Posed Problem Fundamentals,42,0,0,0,64,1,HIGH,0.996403,5


In [33]:
result = generate_adaptive_quiz_priority(
    difficulty,
    telemetry,
    mapping,
    student_id="S24A003",
    course_name="perspectives"
)

display(result)

,StudentID,CourseName,Concept,backward_count,replay_count,study_pause_count,other_work_pause_count,forward_count,speed_increase_count,quiz_priority,confidence,question_count
0,S24A003,perspectives,Algorithm Definition & Properties,1,0,0,0,5,1,LOW,1.0,1
1,S24A003,perspectives,Perspectives Fundamentals,5,0,0,0,7,0,LOW,1.0,1
2,S24A003,perspectives,Search Fundamentals,1,0,0,0,0,0,LOW,1.0,1
